# AnalystLab Africa — Machine Learning Internship Programme
## Week 1: Machine Learning Problem Framing & Data Understanding
### Customer Churn Prediction — ABC Communications Ltd

---

| Field | Details |
|---|---|
| **Author** | Rudolf |
| **Role** | Junior ML Engineer |
| **Dataset** | Telco Customer Churn Dataset (Kaggle) |
| **Week** | Week 1 |
| **Deadline** | Sunday, 11:59 PM WAT |

---

## Notebook Structure

1. Environment Setup & Imports
2. Dataset Loading
3. Dataset Inspection (Part 2)
4. Problem Framing (Part 3)
5. Exploratory Data Analysis — Bar Charts (Part 4)
6. Exploratory Data Analysis — Histograms (Part 4)
7. Exploratory Data Analysis — Correlation Heatmap (Part 4)
8. Model Planning Summary (Part 5)
9. Key Findings & Conclusion

---
## 1. Environment Setup & Imports

In [ ]:
# Core libraries 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

#  Plot styling 
BRAND_BLUE   = '#1F3864'
ACCENT_GOLD  = '#C9A84C'
CHURN_RED    = '#C0392B'
STAY_GREEN   = '#1E6B3C'
PALETTE      = [STAY_GREEN, CHURN_RED]

plt.rcParams.update({
    'figure.facecolor' : 'white',
    'axes.facecolor'   : '#F9F9F9',
    'axes.edgecolor'   : '#CCCCCC',
    'axes.titlesize'   : 14,
    'axes.titleweight' : 'bold',
    'axes.titlecolor'  : BRAND_BLUE,
    'axes.labelsize'   : 12,
    'axes.labelcolor'  : '#333333',
    'xtick.labelsize'  : 10,
    'ytick.labelsize'  : 10,
    'legend.fontsize'  : 10,
    'font.family'      : 'DejaVu Sans',
})

print('Libraries loaded successfully.')
print(f'   pandas  : {pd.__version__}')
print(f'   numpy   : {np.__version__}')
print(f'   seaborn : {sns.__version__}')

---
## 2. Dataset Loading

> Downloaded the dataset from Kaggle and place `WA_Fn-UseC_-Telco-Customer-Churn.csv` in the same folder as this notebook. Then run the cell below.

In [ ]:
# Load the dataset 
CSV_PATH = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'

df = pd.read_csv(CSV_PATH)

print(f'Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

---
## 3. Dataset Inspection (Part 2)
### 3.1 Shape & Column Names

In [ ]:
print('=' * 60)
print('DATASET SHAPE')
print('=' * 60)
print(f'  Rows    : {df.shape[0]:,}')
print(f'  Columns : {df.shape[1]}')
print()
print('COLUMN NAMES')
print('=' * 60)
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2}. {col}')

### 3.2 Data Types

In [ ]:
print('=' * 60)
print('DATA TYPES PER COLUMN')
print('=' * 60)
dtype_df = pd.DataFrame({
    'Column'   : df.columns,
    'Dtype'    : df.dtypes.values,
    'Non-Null' : df.notnull().sum().values,
    'Unique'   : df.nunique().values,
})
print(dtype_df.to_string(index=False))

### 3.3 Missing Values

In [ ]:
# Check for NaN missing values 
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print('No NaN missing values detected in any column.')
else:
    print('Columns with NaN missing values:')
    print(missing_df)

# Check TotalCharges specifically (stored as string, blanks = missing) 
print()
print('=' * 60)
print('TotalCharges dtype check (expected: object / string)')
print('=' * 60)
print(f'  dtype        : {df["TotalCharges"].dtype}')
blank_count = (df['TotalCharges'].str.strip() == '').sum()
print(f'  Blank strings: {blank_count}')
print()
print('NOTE: TotalCharges has blank string entries (not NaN).')
print('      These 11 rows will be treated as missing values.')

### 3.4 Duplicate Records

In [ ]:
print('=' * 60)
print('DUPLICATE RECORDS CHECK')
print('=' * 60)
dup_rows = df.duplicated().sum()
dup_ids  = df['customerID'].duplicated().sum()
print(f'  Duplicate full rows   : {dup_rows}')
print(f'  Duplicate customerIDs : {dup_ids}')
print()
if dup_rows == 0 and dup_ids == 0:
    print('No duplicates found. Dataset has clean, unique customer entries.')
else:
    print('Duplicates detected — investigate before modeling.')

### 3.5 Descriptive Statistics

In [ ]:
# ── Fix TotalCharges first for accurate stats ─────────────────────────────────
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print('NUMERICAL FEATURES — DESCRIPTIVE STATISTICS')
print('=' * 60)
df[['tenure', 'MonthlyCharges', 'TotalCharges']].describe().round(2)

### 3.6 Target Variable Distribution

In [ ]:
print('=' * 60)
print('TARGET VARIABLE: Churn Distribution')
print('=' * 60)
churn_counts = df['Churn'].value_counts()
churn_pct    = df['Churn'].value_counts(normalize=True) * 100
churn_summary = pd.DataFrame({'Count': churn_counts, 'Percentage (%)': churn_pct.round(2)})
print(churn_summary)
print()
print('Class imbalance detected: ~73% No vs ~27% Yes')
print('   Use F1-Score and ROC-AUC as primary metrics, not accuracy.')

---
## 4. Problem Framing (Part 3)
### 4.1 Target Variable & Feature Identification

In [ ]:
# Identify feature categories 
TARGET     = 'Churn'
IDENTIFIER = 'customerID'

NUMERICAL_FEATURES = ['tenure', 'MonthlyCharges', 'TotalCharges']

BINARY_FEATURES = ['SeniorCitizen']

LABEL_ENCODE_FEATURES = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling'
]

ONE_HOT_FEATURES = [
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaymentMethod'
]

ALL_FEATURES = NUMERICAL_FEATURES + BINARY_FEATURES + LABEL_ENCODE_FEATURES + ONE_HOT_FEATURES

print('=' * 60)
print('PROBLEM FRAMING SUMMARY')
print('=' * 60)
print(f'  Task Type        : Binary Classification')
print(f'  Target Variable  : {TARGET}')
print(f'  Identifier (drop): {IDENTIFIER}')
print(f'  Total Features   : {len(ALL_FEATURES)}')
print()
print(f'  Numerical Features   ({len(NUMERICAL_FEATURES)}): {NUMERICAL_FEATURES}')
print(f'  Binary Features      ({len(BINARY_FEATURES)}): {BINARY_FEATURES}')
print(f'  Label Encode         ({len(LABEL_ENCODE_FEATURES)}): {LABEL_ENCODE_FEATURES}')
print(f'  One-Hot Encode       ({len(ONE_HOT_FEATURES)}): {ONE_HOT_FEATURES}')

### 4.2 Preprocessing Requirements

In [ ]:
preprocessing_steps = [
    ('1', 'Drop customerID',                'Not predictive — unique row identifier'),
    ('2', 'Convert TotalCharges to float',  'Currently stored as string due to blank values'),
    ('3', 'Drop 11 blank TotalCharges rows','Only 0.16% of data — minimal information loss'),
    ('4', 'Encode Churn as 1/0',            'Yes → 1, No → 0'),
    ('5', 'Label encode binary categoricals','gender, Partner, Dependents, etc.'),
    ('6', 'One-hot encode multi-class cols', 'Contract, PaymentMethod, InternetService, etc.'),
    ('7', 'Scale numerical features',        'StandardScaler on tenure, MonthlyCharges, TotalCharges'),
    ('8', 'Handle class imbalance',          'class_weight="balanced" or SMOTE'),
    ('9', 'Train/test split',                '80/20 with stratify=y and random_state=42'),
]

print('PREPROCESSING PIPELINE')
print('=' * 70)
print(f'  {"Step":<4}  {"Action":<40}  {"Reason"}')
print('-' * 70)
for step, action, reason in preprocessing_steps:
    print(f'  {step:<4}  {action:<40}  {reason}')

---
## 5. Exploratory Data Analysis — Bar Charts (Part 4)
### Bar Chart 1: Churn Distribution (Target Variable Overview)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

churn_counts = df['Churn'].value_counts()
bars = ax.bar(
    churn_counts.index,
    churn_counts.values,
    color=PALETTE,
    edgecolor='white',
    linewidth=1.5,
    width=0.5
)

# Annotate bars with count and percentage
total = churn_counts.sum()
for bar, val in zip(bars, churn_counts.values):
    pct = val / total * 100
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 50,
        f'{val:,}\n({pct:.1f}%)',
        ha='center', va='bottom',
        fontsize=12, fontweight='bold', color=BRAND_BLUE
    )

ax.set_title('Customer Churn Distribution\n(Target Variable)', pad=15)
ax.set_xlabel('Churn', labelpad=10)
ax.set_ylabel('Number of Customers', labelpad=10)
ax.set_xticklabels(['No (Stayed)', 'Yes (Churned)'])
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_ylim(0, churn_counts.max() * 1.2)
ax.axhline(y=0, color=BRAND_BLUE, linewidth=1)

# Imbalance note
ax.text(0.98, 0.95, '⚠ Class Imbalance: 73% vs 27%',
        transform=ax.transAxes, ha='right', va='top',
        fontsize=9, color='#8B4513',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFF3CD', edgecolor='#C9A84C'))

plt.tight_layout()
plt.savefig('chart1_churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart 1 saved: chart1_churn_distribution.png')

### Bar Chart 2: Churn Rate by Contract Type

In [ ]:
contract_churn = df.groupby('Contract')['Churn'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).reset_index()
contract_churn.columns = ['Contract', 'Churn Rate (%)']
contract_churn = contract_churn.sort_values('Churn Rate (%)', ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))

colors = [CHURN_RED if r > 30 else ACCENT_GOLD if r > 15 else STAY_GREEN
          for r in contract_churn['Churn Rate (%)']]

bars = ax.bar(
    contract_churn['Contract'],
    contract_churn['Churn Rate (%)'],
    color=colors, edgecolor='white', linewidth=1.5, width=0.5
)

for bar, val in zip(bars, contract_churn['Churn Rate (%)']):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.8,
        f'{val:.1f}%',
        ha='center', va='bottom',
        fontsize=13, fontweight='bold', color=BRAND_BLUE
    )

ax.set_title('Churn Rate by Contract Type\n(Key Driver: Month-to-Month customers churn most)', pad=15)
ax.set_xlabel('Contract Type', labelpad=10)
ax.set_ylabel('Churn Rate (%)', labelpad=10)
ax.set_ylim(0, contract_churn['Churn Rate (%)'].max() * 1.25)
ax.axhline(y=0, color=BRAND_BLUE, linewidth=1)

plt.tight_layout()
plt.savefig('chart2_churn_by_contract.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart 2 saved: chart2_churn_by_contract.png')

### Bar Chart 3: Churn Rate by Internet Service Type

In [ ]:
internet_churn = df.groupby('InternetService')['Churn'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).reset_index()
internet_churn.columns = ['InternetService', 'Churn Rate (%)']
internet_churn = internet_churn.sort_values('Churn Rate (%)', ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))

colors2 = [CHURN_RED if r > 35 else ACCENT_GOLD if r > 15 else STAY_GREEN
           for r in internet_churn['Churn Rate (%)']]

bars = ax.bar(
    internet_churn['InternetService'],
    internet_churn['Churn Rate (%)'],
    color=colors2, edgecolor='white', linewidth=1.5, width=0.5
)

for bar, val in zip(bars, internet_churn['Churn Rate (%)']):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.8,
        f'{val:.1f}%',
        ha='center', va='bottom',
        fontsize=13, fontweight='bold', color=BRAND_BLUE
    )

ax.set_title('Churn Rate by Internet Service Type\n(Fiber Optic customers show highest churn)', pad=15)
ax.set_xlabel('Internet Service Type', labelpad=10)
ax.set_ylabel('Churn Rate (%)', labelpad=10)
ax.set_ylim(0, internet_churn['Churn Rate (%)'].max() * 1.25)
ax.axhline(y=0, color=BRAND_BLUE, linewidth=1)

plt.tight_layout()
plt.savefig('chart3_churn_by_internet.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart 3 saved: chart3_churn_by_internet.png')

### Bar Chart 4: Churn Rate by Payment Method

In [ ]:
payment_churn = df.groupby('PaymentMethod')['Churn'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).reset_index()
payment_churn.columns = ['PaymentMethod', 'Churn Rate (%)']
payment_churn = payment_churn.sort_values('Churn Rate (%)', ascending=False)

# Shorten labels for readability
label_map = {
    'Electronic check'             : 'Electronic\nCheck',
    'Mailed check'                 : 'Mailed\nCheck',
    'Bank transfer (automatic)'    : 'Bank Transfer\n(Auto)',
    'Credit card (automatic)'      : 'Credit Card\n(Auto)',
}
payment_churn['PaymentMethod'] = payment_churn['PaymentMethod'].map(
    lambda x: label_map.get(x, x)
)

fig, ax = plt.subplots(figsize=(9, 5))

colors3 = [CHURN_RED if r > 35 else ACCENT_GOLD if r > 20 else STAY_GREEN
           for r in payment_churn['Churn Rate (%)']]

bars = ax.bar(
    payment_churn['PaymentMethod'],
    payment_churn['Churn Rate (%)'],
    color=colors3, edgecolor='white', linewidth=1.5, width=0.5
)

for bar, val in zip(bars, payment_churn['Churn Rate (%)']):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.8,
        f'{val:.1f}%',
        ha='center', va='bottom',
        fontsize=12, fontweight='bold', color=BRAND_BLUE
    )

ax.set_title('Churn Rate by Payment Method\n(Electronic Check users churn at the highest rate)', pad=15)
ax.set_xlabel('Payment Method', labelpad=10)
ax.set_ylabel('Churn Rate (%)', labelpad=10)
ax.set_ylim(0, payment_churn['Churn Rate (%)'].max() * 1.25)
ax.axhline(y=0, color=BRAND_BLUE, linewidth=1)

plt.tight_layout()
plt.savefig('chart4_churn_by_payment.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart 4 saved: chart4_churn_by_payment.png')

---
## 6. Exploratory Data Analysis — Histograms (Part 4)
### Histogram 1: Distribution of Customer Tenure by Churn Status

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

stayed  = df[df['Churn'] == 'No']['tenure']
churned = df[df['Churn'] == 'Yes']['tenure']

ax.hist(stayed,  bins=30, alpha=0.7, color=STAY_GREEN,  label='No Churn (Stayed)',  edgecolor='white')
ax.hist(churned, bins=30, alpha=0.7, color=CHURN_RED,   label='Yes Churn (Left)',   edgecolor='white')

ax.axvline(stayed.mean(),  color=STAY_GREEN,  linestyle='--', linewidth=2, label=f'Mean (Stayed): {stayed.mean():.1f} mo')
ax.axvline(churned.mean(), color=CHURN_RED,   linestyle='--', linewidth=2, label=f'Mean (Churned): {churned.mean():.1f} mo')

ax.set_title('Distribution of Customer Tenure by Churn Status\n(Churned customers leave much earlier)', pad=15)
ax.set_xlabel('Tenure (Months)', labelpad=10)
ax.set_ylabel('Number of Customers', labelpad=10)
ax.legend(framealpha=0.9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Insight annotation
ax.text(0.98, 0.95,
        'Insight: Most churn happens\nin the first 12 months',
        transform=ax.transAxes, ha='right', va='top', fontsize=9,
        color=BRAND_BLUE,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#EEF2FF', edgecolor=BRAND_BLUE))

plt.tight_layout()
plt.savefig('hist1_tenure_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Histogram 1 saved: hist1_tenure_distribution.png')

### Histogram 2: Distribution of Monthly Charges by Churn Status

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

stayed_mc  = df[df['Churn'] == 'No']['MonthlyCharges']
churned_mc = df[df['Churn'] == 'Yes']['MonthlyCharges']

ax.hist(stayed_mc,  bins=30, alpha=0.7, color=STAY_GREEN, label='No Churn (Stayed)', edgecolor='white')
ax.hist(churned_mc, bins=30, alpha=0.7, color=CHURN_RED,  label='Yes Churn (Left)',  edgecolor='white')

ax.axvline(stayed_mc.mean(),  color=STAY_GREEN, linestyle='--', linewidth=2,
           label=f'Mean (Stayed): ${stayed_mc.mean():.2f}')
ax.axvline(churned_mc.mean(), color=CHURN_RED,  linestyle='--', linewidth=2,
           label=f'Mean (Churned): ${churned_mc.mean():.2f}')

ax.set_title('Distribution of Monthly Charges by Churn Status\n(Churned customers pay more per month on average)', pad=15)
ax.set_xlabel('Monthly Charges (USD)', labelpad=10)
ax.set_ylabel('Number of Customers', labelpad=10)
ax.legend(framealpha=0.9)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

ax.text(0.98, 0.95,
        'Insight: Higher charges\ncorrelate with churn',
        transform=ax.transAxes, ha='right', va='top', fontsize=9,
        color=BRAND_BLUE,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#EEF2FF', edgecolor=BRAND_BLUE))

plt.tight_layout()
plt.savefig('hist2_monthly_charges.png', dpi=150, bbox_inches='tight')
plt.show()
print('Histogram 2 saved: hist2_monthly_charges.png')

---
## 7. Exploratory Data Analysis — Correlation Heatmap (Part 4)

In [ ]:
# Encode categoricals for correlation analysis 
df_corr = df.copy()
df_corr['Churn_binary'] = (df_corr['Churn'] == 'Yes').astype(int)

# Simple binary encoding for correlation
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    df_corr[col] = (df_corr[col] == 'Yes').astype(int)

# Select numeric columns for heatmap
numeric_cols = ['SeniorCitizen', 'Partner', 'Dependents', 'tenure',
                'PhoneService', 'PaperlessBilling', 'MonthlyCharges',
                'TotalCharges', 'Churn_binary']

# Rename for cleaner display
rename_map = {
    'SeniorCitizen'   : 'Senior',
    'Partner'         : 'Partner',
    'Dependents'      : 'Dependents',
    'tenure'          : 'Tenure',
    'PhoneService'    : 'Phone',
    'PaperlessBilling': 'Paperless',
    'MonthlyCharges'  : 'Monthly$',
    'TotalCharges'    : 'Total$',
    'Churn_binary'    : 'CHURN',
}

corr_df = df_corr[numeric_cols].rename(columns=rename_map)
corr_matrix = corr_df.corr()

# Plot heatmap 
fig, ax = plt.subplots(figsize=(10, 8))

mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask)] = True  # Show lower triangle only

cmap = sns.diverging_palette(220, 10, as_cmap=True)

sns.heatmap(
    corr_matrix,
    mask=mask,
    cmap=cmap,
    annot=True,
    fmt='.2f',
    linewidths=0.5,
    linecolor='white',
    vmin=-1, vmax=1,
    square=True,
    ax=ax,
    annot_kws={'size': 10, 'weight': 'bold'},
    cbar_kws={'shrink': 0.8, 'label': 'Correlation Coefficient'}
)

ax.set_title('Feature Correlation Heatmap\n(Correlation with Churn — CHURN column)', pad=20)
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right', fontsize=10)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)

plt.tight_layout()
plt.savefig('heatmap_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Heatmap saved: heatmap_correlation.png')

# Print CHURN correlations ranked 
print()
print('CORRELATION WITH CHURN (Ranked)')
print('=' * 40)
churn_corr = corr_matrix['CHURN'].drop('CHURN').sort_values(key=abs, ascending=False)
for feat, val in churn_corr.items():
    direction = '🔴 +' if val > 0 else '🟢 -'
    print(f'  {direction}{abs(val):.3f}  {feat}')

---
## 8. Model Planning Summary (Part 5)

In [ ]:
print('=' * 70)
print('MACHINE LEARNING PROPOSAL SUMMARY')
print('Customer Churn Prediction — ABC Communications Ltd')
print('=' * 70)

print()
print('RECOMMENDED ALGORITHMS (in order of development)')
print('-' * 70)
algorithms = [
    ('1', 'Logistic Regression',  'Baseline model — interpretable and fast'),
    ('2', 'Decision Tree',        'Visualizable rules — good for stakeholder explanation'),
    ('3', 'Random Forest',        'Robust ensemble — good feature importance output'),
    ('4', 'XGBoost',              'High performance — typically best on tabular data'),
    ('5', 'LightGBM',             'Alternative to XGBoost — faster on large datasets'),
]
for rank, name, note in algorithms:
    print(f'  {rank}. {name:<25} — {note}')

print()
print('EVALUATION METRICS')
print('-' * 70)
metrics = [
    ('F1-Score',        'PRIMARY',       'Best metric for imbalanced classification'),
    ('ROC-AUC',         'PRIMARY',       'Target: > 0.80'),
    ('Recall',          'PRIMARY',       'Minimize missed churners'),
    ('Precision',       'SECONDARY',     'Avoid over-alerting retention team'),
    ('Accuracy',        'INFORMATIONAL', 'Reported but not used for selection'),
    ('Confusion Matrix','DIAGNOSTIC',    'TP, FP, TN, FN breakdown'),
]
for metric, priority, note in metrics:
    print(f'  {metric:<20} [{priority:<15}] — {note}')

print()
print('PREPROCESSING STRATEGY')
print('-' * 70)
print('  Encoding  : Label encoding (binary) + One-hot encoding (multi-class)')
print('  Scaling   : StandardScaler on numerical features')
print('  Imbalance : class_weight="balanced" or SMOTE (training set only)')
print('  Split     : 80/20 train/test with stratify=y, random_state=42')

print()
print('=' * 70)
print('Week 1 Analysis Complete — Rudolf | AnalystLab Africa')
print('=' * 70)

---
## 9. Key Findings & Conclusion

### Findings from Exploratory Analysis

| # | Finding | Business Implication |
|---|---|---|
| 1 | **Month-to-month contracts** have the highest churn rate (~43%) | Incentivize longer contract commitments |
| 2 | **Fiber Optic** internet customers churn at ~42% vs DSL at ~19% | Investigate service quality for Fiber customers |
| 3 | **Electronic Check** payment method correlates with high churn | Customers using manual payments may be less committed |
| 4 | **Most churn happens in the first 12 months** of tenure | Focus early retention efforts on new customers |
| 5 | **Churned customers pay more per month** on average | Price sensitivity may be a key churn driver |
| 6 | **Tenure and TotalCharges** are strongly positively correlated (expected) | May need to drop one to avoid multicollinearity |
| 7 | **Monthly charges correlate positively with churn** | High-paying customers need better service to retain |



---
*Notebook authored by Rudolf | AnalystLab Africa ML Internship | Week 1*